In [15]:
import os
import json
import torch
from sklearn.metrics import accuracy_score

from gap.image_loader import PetDataProcessor
from gap.classifiers import CustomCNN, train_cnn, evaluate_cnn
from gap.classifiers import train_logistic_regression_cv, evaluate_classical_model
from gap.utils import set_reproducibility_seeds, save_experiment, get_default_device
from gap.evaluator import ModelEvaluator

# lock seed
SEED = 42
set_reproducibility_seeds(SEED)

# Setup Device
device = get_default_device()
print(f"Running on {device}")

# --- Update this path to your downloaded images folder ---
DATA_DIR = "../datasets/oxford-iiit-pet/images/images"

All random seeds locked to 42.
Running on cuda


In [16]:
print("--- Initializing Data Processor ---")
processor = PetDataProcessor(data_dir=DATA_DIR, random_state=SEED)
classes = processor.encoder.classes_
num_classes = len(classes)
print(f"Loaded {num_classes} pet breeds.\n")

print("--- Preparing Data for PyTorch CNN ---")
# Creates batches of image tensors
IMAGE_SIZE = 224
BATCH_SIZE = 32
train_loader, test_loader = processor.get_cnn_dataloaders(batch_size=BATCH_SIZE, img_size=IMAGE_SIZE)

print("\n--- Preparing Data for Scikit-Learn LRCV ---")
# Creates 1D flattened arrays (downscaled to 64x64 to prevent RAM crashes, but should try larger)
X_train, X_test, y_train, y_test = processor.get_flattened_data(img_size=64)

--- Initializing Data Processor ---
Loaded 37 pet breeds.

--- Preparing Data for PyTorch CNN ---

--- Preparing Data for Scikit-Learn LRCV ---
Flattening images for classical ML (this may take a minute)...


In [17]:
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

class TorchLogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)


def train_logistic_regression_cv_gpu(X, y, cv_folds=3, max_iter=200, lr=1e-3, device="cuda"):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    y = torch.tensor(y, dtype=torch.long).to(device)

    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    models = []
    fold_accuracies = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = TorchLogisticRegression(X.shape[1], len(torch.unique(y))).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        for _ in range(max_iter):
            opt.zero_grad()
            logits = model(X_tr)
            loss = loss_fn(logits, y_tr)
            loss.backward()
            opt.step()

        with torch.no_grad():
            preds = model(X_val).argmax(dim=1)
            acc = accuracy_score(y_val.cpu(), preds.cpu())

        fold_accuracies.append(acc)
        models.append(model)

    # pick best model
    best_idx = int(torch.tensor(fold_accuracies).argmax())
    return models[best_idx]

In [18]:
print("==================================================")
print(" EXPERIMENT 1: Logistic Regression (Cross-Validated)")
print("==================================================")

# train (GPU version)
max_iter = 200
cv_folds = 3
lrcv_model = train_logistic_regression_cv_gpu(
    X_train, y_train, cv_folds=cv_folds, max_iter=max_iter, device=device
)

# evaluate
print("\nEvaluating LRCV Test Performance...")
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    preds = lrcv_model(X_test_t).argmax(dim=1).cpu().numpy()

lrcv_y_true, lrcv_y_pred = y_test, preds

# save
lrcv_config = {
    "model_type": "TorchLogisticRegressionCV",
    "cv_folds": cv_folds,
    "max_iter": max_iter,
    "image_size_flattened": 64,
    "random_seed": SEED,
    "test_accuracy": accuracy_score(lrcv_y_true, lrcv_y_pred),
    "classes": list(classes)
}
save_experiment(lrcv_model, model_name="Torch_LRCV_baseline", config=lrcv_config)

 EXPERIMENT 1: Logistic Regression (Cross-Validated)

Evaluating LRCV Test Performance...
Model and metadata saved to: ../saved_models\Torch_LRCV_baseline_20260308_1211/


'../saved_models\\Torch_LRCV_baseline_20260308_1211'

In [19]:
print("==================================================")
print(" EXPERIMENT 2: Custom PyTorch CNN")
print("==================================================")

# initialize
#cnn_model = CustomCNN(num_classes=num_classes, input_size=IMAGE_SIZE)
cnn_model = CustomCNN(num_classes=num_classes)
EPOCHS = 50
LEARNING_RATE = 0.001

# train
print(f"Training CNN for {EPOCHS} epochs...")
trained_cnn, history = train_cnn(
    cnn_model, train_loader, test_loader, EPOCHS, LEARNING_RATE, device
)

# evaluate
print("\nEvaluating CNN Test Performance...")
cnn_y_true, cnn_y_pred = evaluate_cnn(trained_cnn, test_loader, device=device)

# save
cnn_config = {
    "model_type": "CustomCNN",
    "framework": "PyTorch",
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "random_seed": SEED,
    "test_accuracy": accuracy_score(cnn_y_true, cnn_y_pred),
    "classes": list(classes)
}
save_experiment(trained_cnn, model_name="CustomCNN_10Epochs", config=cnn_config)

 EXPERIMENT 2: Custom PyTorch CNN
Training CNN for 50 epochs...
Starting Training for 50 Epochs...

Epoch [01/50] | Train Loss: 3.4569 Acc: 0.0847 | Val Loss: 3.2828 Acc: 0.1022
   Validation accuracy improved to 0.1022!
Epoch [02/50] | Train Loss: 3.0860 Acc: 0.1494 | Val Loss: 3.2649 Acc: 0.1475
   Validation accuracy improved to 0.1475!
Epoch [03/50] | Train Loss: 2.8014 Acc: 0.2018 | Val Loss: 2.8389 Acc: 0.1955
   Validation accuracy improved to 0.1955!
Epoch [04/50] | Train Loss: 2.5343 Acc: 0.2703 | Val Loss: 2.8566 Acc: 0.1922
Epoch [05/50] | Train Loss: 2.3009 Acc: 0.3336 | Val Loss: 2.3282 Acc: 0.3315
   Validation accuracy improved to 0.3315!
Epoch [06/50] | Train Loss: 2.0664 Acc: 0.3828 | Val Loss: 2.5452 Acc: 0.2963
Epoch [07/50] | Train Loss: 1.8817 Acc: 0.4283 | Val Loss: 2.2690 Acc: 0.3539
   Validation accuracy improved to 0.3539!
Epoch [08/50] | Train Loss: 1.7182 Acc: 0.4777 | Val Loss: 2.0536 Acc: 0.3924
   Validation accuracy improved to 0.3924!
Epoch [09/50] | Tr

'../saved_models\\CustomCNN_10Epochs_20260308_1245'

In [24]:
weights_path = '../saved_models/CustomCNN_10Epochs_20260308_1245/weights.pth'

lrcv_model = TorchLogisticRegression(input_dim=X_train.shape[1], num_classes=len(classes))
lrcv_model.load_state_dict(torch.load(weights_path, map_location=device))
lrcv_model.to(device)
lrcv_model.eval()

RuntimeError: Error(s) in loading state_dict for TorchLogisticRegression:
	Missing key(s) in state_dict: "linear.weight", "linear.bias". 
	Unexpected key(s) in state_dict: "features.0.weight", "features.0.bias", "features.1.weight", "features.1.bias", "features.1.running_mean", "features.1.running_var", "features.1.num_batches_tracked", "features.4.weight", "features.4.bias", "features.5.weight", "features.5.bias", "features.5.running_mean", "features.5.running_var", "features.5.num_batches_tracked", "features.8.weight", "features.8.bias", "features.9.weight", "features.9.bias", "features.9.running_mean", "features.9.running_var", "features.9.num_batches_tracked", "features.12.weight", "features.12.bias", "features.13.weight", "features.13.bias", "features.13.running_mean", "features.13.running_var", "features.13.num_batches_tracked", "features.16.weight", "features.16.bias", "features.17.weight", "features.17.bias", "features.17.running_mean", "features.17.running_var", "features.17.n

In [25]:
evaluator = ModelEvaluator(loaded_model, test_loader, classes, device)


evaluator.generate_full_report()

NameError: name 'loaded_model' is not defined